In [1]:
# import features
import pandas as pd
# acitvity entropy
df_entropy = pd.read_csv('../output/data_cleaned/activity_entropy_rates.csv')
print(df_entropy.shape)

# early warning scores (physiological)
df_EWS = pd.read_csv('../output/data_cleaned/Mian_warning_score.csv')
df_EWS.columns = ['patient_id', 'date', 'early_warning_score']


# sleep quality
df_sleep_quality = pd.read_csv('../output/sleep_score/sleep_quality_score_by_duration.csv')
df_sleep_quality = df_sleep_quality.drop(columns= ['sum', 'mean','scaled_sleep_quality_sum']).copy()
df_sleep_quality.columns = ['patient_id', 'date', 'sleep_quality_score']

# agitation
df_agitation = pd.read_csv('../output/data_cleaned/agitation_daily_counts.csv')

# uti
df_uti = pd.read_csv('../output/data_cleaned/uti_daily.csv')

merged_df = df_entropy
# merge dataframes
for df in [df_EWS, df_sleep_quality, df_agitation, df_uti]:
    print(df.shape)
    merged_df = pd.merge(merged_df, df, on=['patient_id', 'date'], how='outer')
    
# only consider the patients without NA in the following analysis (as the analysis itself will be individualized anyway)
analysis_df = merged_df.dropna(subset=['sleep_quality_score']).dropna(subset=['early_warning_score']).dropna(subset=['entropy_rate']).copy()
analysis_df =  analysis_df.fillna(0)

print(analysis_df.shape)
analysis_df 

(2722, 3)
(2160, 3)
(800, 3)
(115, 3)
(265, 3)
(660, 7)


,patient_id,date,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen
219,0f352,2019-06-26,0.669008,0.0,2.233437,0.0,1.0
221,0f352,2019-06-28,0.613697,0.0,1.516700,0.0,0.0
222,0f352,2019-06-29,0.615494,0.0,-0.010261,0.0,1.0
223,0f352,2019-06-30,0.514768,0.0,2.607387,0.0,0.0
250,16f4b,2019-04-28,0.651879,0.0,-1.319084,0.0,0.0
...,...,...,...,...,...,...,...
2694,f220c,2019-06-06,0.620975,0.0,-2.627908,0.0,0.0
2696,f220c,2019-06-08,0.527123,0.0,2.607387,0.0,0.0
2706,f220c,2019-06-19,0.527442,0.0,2.607387,0.0,0.0
2709,f220c,2019-06-22,0.608841,0.0,2.607387,0.0,0.0


In [88]:
id_select = "ec812"
df_person = analysis_df[analysis_df['patient_id'] == id_select].copy()

df_person['date'] = pd.to_datetime(df_person['date']).dt.strftime('%m-%d').astype(str).values
df_person

,patient_id,date,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen
2528,ec812,04-09,0.684974,0.0,-0.010261,0.0,0.0
2529,ec812,04-10,0.614688,0.0,0.316945,0.0,0.0
2530,ec812,04-11,0.680348,0.0,-0.446535,0.0,0.0
2531,ec812,04-12,0.668105,0.0,-0.337467,0.0,0.0
2532,ec812,04-13,0.655223,0.0,0.363689,0.0,0.0
...,...,...,...,...,...,...,...
2594,ec812,06-21,0.661363,0.0,-0.337467,0.0,0.0
2595,ec812,06-22,0.623560,0.0,0.862288,0.0,0.0
2596,ec812,06-23,0.708682,0.0,0.316945,0.0,0.0
2597,ec812,06-24,0.656330,0.0,0.280589,0.0,0.0


In [93]:
df_anomaly_record = pd.read_csv('../output/Anomaly_delirium/forest_isolation_sliding_0.1contamination/forest_isolation_anomaly_data.csv')
df_merged = df_person.merge(df_anomaly_record[['patient_id', 'date']], on=['patient_id', 'date'], how='left', indicator=True)
df_merged['anomaly_flags'] = np.where(df_merged['_merge'] == 'both', True, False)
anomaly_flags_all = df_merged['anomaly_flags'].values

anomaly_flags_all

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False,  True, False, False, False,  True,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False,  True, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False])

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Sort by patient_id and date
patient_data = df_person.sort_values(by=['patient_id', 'date']).reset_index(drop=True)

features = [
    'entropy_rate',
    'early_warning_score',
    'sleep_quality_score',
    'agitation_counts',
    'uti_happen'
]

previous_n = 10  # number of past days used as baseline

X_list = []
y_list = []
scalers = []  # optional: store scalers for inspection/debugging

for i in range(len(patient_data) - previous_n):
    
    # ---------- historical window ----------
    X_raw = patient_data.iloc[i:i+previous_n][features].values
    
    # ---------- fit scaler on historical data only ----------
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    
    # ---------- next-day observation ----------
    y_raw = patient_data.iloc[i+previous_n][features].values.reshape(1, -1)
    y_scaled = scaler.transform(y_raw)
    
    X_list.append(X_scaled)
    y_list.append(y_scaled.squeeze())  # shape (n_features,)
    scalers.append(scaler)

# Convert to arrays
X = np.array(X_list)  # shape: (n_windows, previous_n, n_features)
y = np.array(y_list)  # shape: (n_windows, n_features)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (60, 10, 5)
y shape: (60, 5)


In [109]:
from sklearn.ensemble import IsolationForest

features = [
    'entropy_rate',
    'early_warning_score',
    'sleep_quality_score',
    'agitation_counts',
    'uti_happen'
]

# Parameters
n_estimators = 100  # Number of trees
contamination = 0.1  # Expected proportion of anomalies: I want to use the previous days as baseline, so expect no/almost no anomaly here 
max_samples=0.8  # Number of samples used to train each tree (here I use fraction of total sample)

# Train Isolation Forest
iso_forest = IsolationForest(n_estimators=n_estimators,
                            contamination=contamination,
                            max_samples=max_samples,
                            random_state=42)



importances_list = []
X_anomaly_list = []
y_anomaly_list = []

anomaly_flags = anomaly_flags_all[previous_n:]
for n_input in range(len(X)): 
    if anomaly_flags[n_input]:    
        iso_forest.fit(X[n_input])
        
        X_anomaly_list.append(iso_forest.predict(X[n_input]))
        y_anomaly_list.append(iso_forest.predict(y[n_input].reshape(1, -1)))
        
        X_anomaly = np.array(X_anomaly_list)
        y_anomaly = np.array(y_anomaly_list)
        
        # df_anomaly_combined = pd.concat([pd.DataFrame(X_anomaly[0,]), pd.DataFrame(y_anomaly)], axis=0)
        df_anomaly_combined = pd.concat([pd.DataFrame([np.nan]*previous_n), pd.DataFrame(y_anomaly)], axis=0)
        df_anomaly_combined = df_anomaly_combined.rename(columns={df_anomaly_combined.columns[0]: "anomaly_FI"})      
           
        
        base_score = iso_forest.decision_function(y[n_input].reshape(1, -1))[0]
    
        
        # permutation 
        importances = {}
        shuffle_times = 50
        for j, fname in enumerate(features):
            X_perm = X[n_input].copy()
            
            perm_score_avg_list = []
            for shuffle_n in range(shuffle_times):
                np.random.seed(shuffle_n)
                np.random.shuffle(X_perm[:, j])
        
                model_perm = IsolationForest(n_estimators=n_estimators,
                                    contamination=contamination,
                                    max_samples=max_samples,
                                    random_state=42)
                model_perm.fit(X_perm)
        
                perm_score = model_perm.decision_function(y[n_input].reshape(1, -1))[0]
                perm_score_avg_list.append(perm_score)
            perm_score_avg = np.mean(perm_score_avg_list)
            importances[fname] = base_score - perm_score_avg
        
        importances_list.append(importances)
    
importances_df = pd.DataFrame(importances_list)    
importances_df['patient_id'] = id_select       


# ALL in ONE

In [174]:
df_anomaly_record = pd.read_csv('../output/Anomaly_delirium/forest_isolation_sliding_0.1contamination/forest_isolation_anomaly_data.csv')

days_baseline = 10

features = [
    'entropy_rate',
    'early_warning_score',
    'sleep_quality_score',
    'agitation_counts',
    'uti_happen'
]

importances_df_list = []        
for id_select in  df_anomaly_record['patient_id'].unique():
    df_person = analysis_df[analysis_df['patient_id'] == id_select].copy()
    df_person['date'] = pd.to_datetime(df_person['date']).dt.strftime('%m-%d').astype(str).values
    if df_person.shape[0] > days_baseline:
        print(id_select)
        
        # only run permutation on anomaly windows
        df_merged = df_person.merge(df_anomaly_record[['patient_id', 'date']], on=['patient_id', 'date'], how='left', indicator=True).copy()
        df_merged['anomaly_flags'] = np.where(df_merged['_merge'] == 'both', True, False)
        anomaly_flags_all = df_merged['anomaly_flags'].values


        # Sort by patient_id and date
        patient_data = df_person.sort_values(by=['patient_id', 'date']).reset_index(drop=True)
         
        previous_n = days_baseline  # number of past days used as baseline
        
        X_list = []
        y_list = []
        scalers = []  # optional: store scalers for inspection/debugging
        
        for i in range(len(patient_data) - previous_n):
            
            # ---------- historical window ----------
            X_raw = patient_data.iloc[i:i+previous_n][features].values
            
            # ---------- fit scaler on historical data only ----------
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X_raw)
            
            # ---------- next-day observation ----------
            y_raw = patient_data.iloc[i+previous_n][features].values.reshape(1, -1)
            y_scaled = scaler.transform(y_raw)
            
            X_list.append(X_scaled)
            y_list.append(y_scaled.squeeze())  # shape (n_features,)
            scalers.append(scaler)
        
        # Convert to arrays
        X = np.array(X_list)  # shape: (n_windows, previous_n, n_features)
        y = np.array(y_list)  # shape: (n_windows, n_features)   
        
        
        
         # Parameters
        n_estimators = 100  # Number of trees
        contamination = 0.1  # Expected proportion of anomalies: I want to use the previous days as baseline, so expect no/almost no anomaly here 
        max_samples=0.8  # Number of samples used to train each tree (here I use fraction of total sample)
        
        # Train Isolation Forest
        iso_forest = IsolationForest(n_estimators=n_estimators,
                                    contamination=contamination,
                                    max_samples=max_samples,
                                    random_state=42)
        
        

        importances_list = []      
        anomaly_flags = anomaly_flags_all[previous_n:]
        for n_input in range(len(X)): 
            if anomaly_flags[n_input]:
                iso_forest.fit(X[n_input])
                
                base_score = iso_forest.decision_function(y[n_input].reshape(1, -1))[0]
            
                
                # permutation 
                importances = {}
                shuffle_times = 50
                for j, fname in enumerate(features):
                    X_perm = X[n_input].copy()
                    
                    perm_score_avg_list = []
                    for shuffle_n in range(shuffle_times):
                        np.random.seed(shuffle_n)
                        np.random.shuffle(X_perm[:, j])
                
                        model_perm = IsolationForest(n_estimators=n_estimators,
                                            contamination=contamination,
                                            max_samples=max_samples,
                                            random_state=42)
                        model_perm.fit(X_perm)
                
                        perm_score = model_perm.decision_function(y[n_input].reshape(1, -1))[0]
                        perm_score_avg_list.append(perm_score)
                        
                    perm_score_avg = np.mean(perm_score_avg_list)
                    importances[fname] = base_score - perm_score_avg
                
                importances_list.append(importances)
            
        importances_df = pd.DataFrame(importances_list)    
        importances_df['patient_id'] = id_select         
        importances_df_list.append(importances_df)    

1fbe4
30a32
55cd4
93c14
96adf
a2849
c55f8
c5785
c8574
ec812
f220c


In [178]:
df_all = pd.concat(importances_df_list, axis=0, ignore_index=True)
df_all.to_csv("../output/Anomaly_delirium/feature_importance/feature_importance_raw.csv", index=False)
df_all

,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen,patient_id
0,-0.012508,3.701126e-03,-0.018247,0.000000e+00,-1.892446e-03,1fbe4
1,-0.007061,-5.494390e-03,-0.002746,0.000000e+00,-3.432917e-03,1fbe4
2,-0.018111,-5.239811e-03,-0.018505,0.000000e+00,-8.558744e-03,1fbe4
3,0.005155,2.684636e-02,0.013451,-3.469447e-18,2.443361e-02,1fbe4
4,-0.002346,9.084980e-03,-0.019136,0.000000e+00,7.515919e-03,1fbe4
...,...,...,...,...,...,...
66,-0.012768,-2.322444e-02,-0.013077,0.000000e+00,-1.467471e-02,c8574
67,0.015665,-3.469447e-18,0.015436,-3.469447e-18,-3.469447e-18,ec812
68,0.029951,6.074757e-02,0.003753,3.469447e-18,4.719856e-02,ec812
69,-0.025345,0.000000e+00,-0.021927,-1.580825e-03,0.000000e+00,ec812
